# Лабораторная работа №5 (№9 по практикуму)
## «Нейронная сеть Хопфилда»

**Выполнил:** студент гр. ИТП-41 Стасев Б.В.
**Принял:** ассистент Ястребов А.А.

**Цель работы:** изучение топологии и алгоритма функционирования сети Хопфилда.

**Задание:**
1. Ознакомиться с теоретической частью.
2. Написать программу, реализующую нейронную сеть Хопфилда.
3. Обучить сеть на 3 образа — первые буквы фамилии **Стасев: «С», «Т», «А»** (бинарные изображения 10×10).
4. Подать на вход сети тестовые образы с зашумлением 10, 20, 30, 35, 40, 45, 50, 60, 70, 80, 90, 100 % — не менее 10 образов для каждого класса при каждом проценте зашумления.
5. Проанализировать, при каком проценте зашумления образы распознаются верно.

> Запуск в Google Colab: «Среда выполнения» → «Выполнить все» (Ctrl+F9). Дополнительные библиотеки не нужны.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 110
pd.set_option('display.width', 200)

SEED = 16                     # фиксируем генератор для воспроизводимости
rng = np.random.default_rng(SEED)

### 1. Эталонные образы

Каждая буква — бинарное изображение 10×10 ($n = 100$ нейронов). Чёрная клетка кодируется $+1$, белая $-1$ (биполярный вектор).

In [ ]:
LETTERS_TXT = {
    'С': """
..######..
.########.
###....###
##........
##........
##........
##........
###....###
.########.
..######..""",
    'Т': """
##########
##########
....##....
....##....
....##....
....##....
....##....
....##....
....##....
....##....""",
    'А': """
....##....
...####...
..##..##..
..##..##..
.##....##.
.########.
.########.
##......##
##......##
##......##""",
}

SIZE = (10, 10)


def to_vector(txt):
    """Текстовая картинка -> биполярный вектор (# -> +1, . -> -1)."""
    rows = txt.strip().split('\n')
    return np.array([1 if ch == '#' else -1 for row in rows for ch in row])


NAMES = list(LETTERS_TXT)
PATTERNS = np.array([to_vector(t) for t in LETTERS_TXT.values()])
n = PATTERNS.shape[1]
print('Образы:', NAMES, '| размер матрицы образов:', PATTERNS.shape)


def show(ax, v, title=''):
    """Отрисовка образа: +1 — чёрная клетка, -1 — белая."""
    ax.imshow(v.reshape(SIZE), cmap='gray_r', vmin=-1, vmax=1)
    ax.set_xticks(np.arange(-0.5, SIZE[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, SIZE[0], 1), minor=True)
    ax.grid(which='minor', color='gray', linewidth=0.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.tick_params(which='minor', length=0)
    ax.set_title(title, fontsize=10)


fig, axes = plt.subplots(1, 3, figsize=(7, 2.6))
for ax, name, p in zip(axes, NAMES, PATTERNS):
    show(ax, p, f'«{name}»')
fig.suptitle('Эталонные образы для запоминания')
plt.tight_layout()
plt.show()

### 2. Реализация сети Хопфилда

**Обучение** (правило Хебба, формула 4.1):
$$w_{ij} = \begin{cases}\sum_{k=1}^{m} a_i^k a_j^k, & i \ne j,\\ 0, & i = j.\end{cases}$$

**Воспроизведение** (формулы 4.3, 4.4):
$$a_i(t+1) = f\Big(\sum_{j=1}^{n} w_{ij} a_j(t)\Big),\qquad f(x) = \begin{cases}1, & x > 0,\\ -1, & x \le 0.\end{cases}$$

* **синхронный режим** — на каждом такте все нейроны меняют состояние одновременно;
* **асинхронный режим** — состояние меняет один (случайно выбранный) нейрон, нейроны перебираются в случайном порядке.

Итерации продолжаются, пока сеть не придёт в стабильное состояние. Энергия сети $E = -\frac{1}{2}\sum_i\sum_j w_{ij} a_i a_j$ в асинхронном режиме не возрастает.

In [ ]:
class HopfieldNetwork:
    def __init__(self, n):
        self.n = n
        self.W = np.zeros((n, n))

    def train(self, patterns):
        """Обучение по правилу Хебба (формула 4.1)."""
        self.W = np.zeros((self.n, self.n))
        for a in patterns:
            self.W += np.outer(a, a)
        np.fill_diagonal(self.W, 0)          # нет обратной связи нейрона на себя

    @staticmethod
    def f(x):
        """Биполярная пороговая функция активации (формула 4.4)."""
        return np.where(x > 0, 1, -1)

    def energy(self, a):
        return -0.5 * a @ self.W @ a

    def recall_sync(self, a0, max_iter=100):
        """Синхронный режим: все нейроны обновляются одновременно."""
        a, prev = a0.copy(), None
        for t in range(1, max_iter + 1):
            new = self.f(self.W @ a)
            if np.array_equal(new, a):
                return new, t, 'устойчивое состояние'
            if prev is not None and np.array_equal(new, prev):
                return new, t, 'цикл длины 2'
            prev, a = a, new
        return a, max_iter, 'нет сходимости'

    def recall_async(self, a0, rng, max_sweeps=100, trace=False):
        """Асинхронный режим: за один шаг меняется один нейрон
        (нейроны перебираются в случайном порядке)."""
        a = a0.copy()
        energies = [self.energy(a)]
        for sweep in range(1, max_sweeps + 1):
            changed = False
            for i in rng.permutation(self.n):
                new_i = 1 if self.W[i] @ a > 0 else -1
                if new_i != a[i]:
                    a[i] = new_i
                    changed = True
                if trace:
                    energies.append(self.energy(a))
            if not changed:
                return (a, sweep, 'устойчивое состояние') + ((energies,) if trace else ())
        return (a, max_sweeps, 'нет сходимости') + ((energies,) if trace else ())


net = HopfieldNetwork(n)
net.train(PATTERNS)

print('Матрица весов W:', net.W.shape)
print('Симметричность (w_ij = w_ji):', np.array_equal(net.W, net.W.T))
print('Нулевая главная диагональ (w_ii = 0):', not np.any(np.diag(net.W)))
print('Фрагмент W[0:8, 0:8]:')
print(net.W[:8, :8].astype(int))

In [ ]:
# Максимальное число запоминаемых образов (формула 4.5)
m_max = n / (2 * np.log(n) + np.log(np.log(n)))
print(f'n = {n}: m_max = n / (2 ln n + ln ln n) = {m_max:.2f}; запоминаем m = {len(PATTERNS)} образа')

# Скалярные произведения образов (чем ближе к 0, тем меньше образы мешают друг другу)
print('\nПопарные скалярные произведения образов:')
print(pd.DataFrame(PATTERNS @ PATTERNS.T, index=NAMES, columns=NAMES))

# Проверка: каждый эталон является устойчивым состоянием сети
for name, p in zip(NAMES, PATTERNS):
    print(f'«{name}»: f(W·a) == a ->', np.array_equal(net.f(net.W @ p), p))

fig, ax = plt.subplots(figsize=(4.6, 4))
im = ax.imshow(net.W, cmap='coolwarm')
plt.colorbar(im, ax=ax)
ax.set_title('Матрица весовых коэффициентов W')
plt.tight_layout()
plt.show()

### 3. Зашумление и распознавание

Зашумление на $p\,\%$ — инвертирование $\mathrm{round}(n \cdot p / 100)$ случайно выбранных пикселей эталона (при 100 % получается полный негатив образа).

Образ считается распознанным, если сеть пришла в состояние, **точно совпадающее** с эталоном своего класса.

In [ ]:
NOISE_LEVELS = [10, 20, 30, 35, 40, 45, 50, 60, 70, 80, 90, 100]


def add_noise(p, percent, rng):
    """Инвертирует percent % случайно выбранных пикселей."""
    a = p.copy()
    k = int(round(len(p) * percent / 100))
    idx = rng.choice(len(p), size=k, replace=False)
    a[idx] *= -1
    return a


def identify(state):
    """Какому эталону соответствует состояние сети."""
    for name, p in zip(NAMES, PATTERNS):
        if np.array_equal(state, p):
            return name
        if np.array_equal(state, -p):
            return f'негатив «{name}»'
    return 'ложный образ'

#### 3.1. Примеры тестируемых и искажённых образов (аналог табл. 4.1)

Для каждой буквы при каждом проценте зашумления показана пара: искажённый образ (подаётся на вход) и результат работы сети в асинхронном режиме (✓ — распознан верно, ✗ — неверно).

In [ ]:
ex_rng = np.random.default_rng(SEED)
PAIRS_IN_ROW = 3
for k, name in enumerate(NAMES):
    n_rows = len(NOISE_LEVELS) // PAIRS_IN_ROW
    fig, axes = plt.subplots(n_rows, 1 + 2 * PAIRS_IN_ROW, figsize=(10, 2.0 * n_rows),
                             gridspec_kw={'width_ratios': [1.15] + [1] * (2 * PAIRS_IN_ROW)})
    for ax in axes[:, 0]:
        ax.axis('off')
    show(axes[0, 0], PATTERNS[k], 'Эталон')
    axes[0, 0].axis('on')
    for j, level in enumerate(NOISE_LEVELS):
        noisy = add_noise(PATTERNS[k], level, ex_rng)
        out, _, _ = net.recall_async(noisy, ex_rng)
        res = identify(out)
        r, c = divmod(j, PAIRS_IN_ROW)
        show(axes[r, 1 + 2 * c], noisy, f'Шум {level}%')
        show(axes[r, 2 + 2 * c], out, ('✓ ' if res == name else '✗ ') + res)
        axes[r, 2 + 2 * c].title.set_fontsize(8)
    fig.suptitle(f'Образ «{name}»: искажённый образ -> результат восстановления')
    plt.tight_layout()
    plt.show()

#### 3.2. Основной эксперимент: 10 тестовых образов на класс для каждого процента зашумления

Каждый искажённый образ подаётся на сеть в синхронном и в асинхронном режимах. В таблице — процент правильно распознанных образов.

In [ ]:
def experiment(n_tests, rng):
    rows, outcomes = [], []
    for level in NOISE_LEVELS:
        row = {'Зашумление, %': level}
        for k, name in enumerate(NAMES):
            ok_s = ok_a = 0
            for _ in range(n_tests):
                noisy = add_noise(PATTERNS[k], level, rng)
                s_out, _, s_status = net.recall_sync(noisy)
                a_out, _, _ = net.recall_async(noisy, rng)
                res_s, res_a = identify(s_out), identify(a_out)
                ok_s += res_s == name
                ok_a += res_a == name
                kind = ('верно' if res_a == name else
                        'негатив эталона' if res_a.startswith('негатив') else
                        'ложный образ' if res_a == 'ложный образ' else 'другая буква')
                outcomes.append({'Зашумление, %': level, 'Буква': name, 'Итог': kind,
                                 'Синхр. режим': s_status})
            row[f'«{name}» синхр.'] = 100 * ok_s / n_tests
            row[f'«{name}» асинхр.'] = 100 * ok_a / n_tests
        row['Среднее синхр.'] = np.mean([row[f'«{nm}» синхр.'] for nm in NAMES])
        row['Среднее асинхр.'] = np.mean([row[f'«{nm}» асинхр.'] for nm in NAMES])
        rows.append(row)
    return pd.DataFrame(rows).set_index('Зашумление, %').round(1), pd.DataFrame(outcomes)


N_TESTS = 10
table10, outcomes10 = experiment(N_TESTS, np.random.default_rng(SEED))
print(f'Процент верно распознанных образов ({N_TESTS} тестов на букву для каждого уровня шума):')
display(table10)

In [ ]:
# Что получилось на выходе сети (асинхронный режим), по всем буквам
summary = outcomes10.groupby(['Зашумление, %', 'Итог']).size().unstack(fill_value=0)
summary = summary.reindex(columns=['верно', 'другая буква', 'ложный образ', 'негатив эталона'], fill_value=0)
print('Исходы распознавания (асинхронный режим, 3 буквы × 10 тестов):')
display(summary)

#### 3.3. Расширенный эксперимент: 200 тестов на класс (сглаженные графики)

In [ ]:
table200, outcomes200 = experiment(200, np.random.default_rng(SEED + 1))
display(table200)

print()
print('Состояния, в которые пришла сеть в синхронном режиме (3 буквы × 200 тестов):')
display(outcomes200.groupby(['Зашумление, %', 'Синхр. режим']).size().unstack(fill_value=0))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, mode in zip(axes, ['синхр.', 'асинхр.']):
    for name in NAMES:
        ax.plot(NOISE_LEVELS, table200[f'«{name}» {mode}'], 'o-', label=f'«{name}»')
    ax.plot(NOISE_LEVELS, table10[f'Среднее {mode}'], 'k--', alpha=0.6, label='среднее (10 тестов)')
    ax.set_title(('Синхронный' if mode == 'синхр.' else 'Асинхронный') + ' режим')
    ax.set_xlabel('Зашумление, %')
    ax.set_xticks(NOISE_LEVELS)
    ax.grid(alpha=0.3)
    ax.legend()
axes[0].set_ylabel('Верно распознано, %')
fig.suptitle('Зависимость качества распознавания от процента зашумления (200 тестов на букву)')
plt.tight_layout()
plt.show()

#### 3.4. Энергия сети в процессе восстановления

In [ ]:
e_rng = np.random.default_rng(SEED)
fig, ax = plt.subplots(figsize=(7, 3.6))
for name, p in zip(NAMES, PATTERNS):
    noisy = add_noise(p, 40, e_rng)
    out, sweeps, status, en = net.recall_async(noisy, e_rng, trace=True)
    ax.plot(en, label=f'«{name}», 40 % шума -> {identify(out)}')
ax.axhline(net.energy(PATTERNS[0]), color='gray', ls=':', label='энергия эталона «С»')
ax.set_xlabel('Номер шага (обновление одного нейрона)')
ax.set_ylabel('Энергия E')
ax.set_title('Асинхронный режим: энергия не возрастает')
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 4. Интерактивная проверка (Colab)

Выберите букву, процент шума и режим работы — сеть восстановит образ.

In [ ]:
def demo(letter='С', noise=30, mode='асинхронный', seed=0):
    r = np.random.default_rng(seed)
    k = NAMES.index(letter)
    noisy = add_noise(PATTERNS[k], noise, r)
    if mode == 'синхронный':
        out, it, status = net.recall_sync(noisy)
    else:
        out, it, status = net.recall_async(noisy, r)
    fig, axes = plt.subplots(1, 3, figsize=(7, 2.6))
    show(axes[0], PATTERNS[k], 'Эталон')
    show(axes[1], noisy, f'Шум {noise}%')
    show(axes[2], out, f'Выход: {identify(out)}')
    fig.suptitle(f'{mode} режим: {it} итер., {status}', fontsize=10)
    plt.tight_layout()
    plt.show()


try:
    import ipywidgets as widgets
    widgets.interact(demo,
                     letter=widgets.Dropdown(options=NAMES, description='Буква'),
                     noise=widgets.IntSlider(value=30, min=0, max=100, step=5, description='Шум, %'),
                     mode=widgets.RadioButtons(options=['асинхронный', 'синхронный'], description='Режим'),
                     seed=widgets.IntSlider(value=0, min=0, max=50, description='seed'))
except ImportError:
    demo('А', 30)

### Вывод

В ходе лабораторной работы изучены топология и алгоритм функционирования сети Хопфилда. Реализована сеть из 100 нейронов, обученная по правилу Хебба на образы букв «С», «Т», «А» размером 10×10 (матрица весов симметрична, главная диагональ нулевая, все эталоны — устойчивые состояния). Количество запоминаемых образов (3) не превышает предельного $m_{max} \approx 9{,}3$.

При зашумлении до 30 % все тестовые образы распознаются верно (в расширенном эксперименте из 200 тестов — не менее 96 %), при 35 % — около 90 %, при 40 % — 70–80 %, при 45 % — лишь 30–50 %. При 50 % и более верное распознавание практически невозможно: искажённый образ ближе к **негативу** эталона, чем к самому эталону, и сеть приходит к негативу (инверсный образ — тоже устойчивое состояние сети Хопфилда, так как $E(-a) = E(a)$), к другой букве или к ложному образу. При 100 % зашумления на вход подаётся точный негатив эталона, который сразу является устойчивым состоянием. Синхронный и асинхронный режимы дают близкие результаты; асинхронный режим всегда сходится к устойчивому состоянию, а синхронный при 50 % зашумления в 17 случаях из 600 попал в цикл длины 2.